In [3]:
from pyspark.sql.types import StructType, StructField, StringType,  IntegerType
from pyspark.sql.functions import col, count 

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 6, Finished, Available, Finished, False)

In [4]:
cust_schema = StructType([
    StructField("CustomerID", StringType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("MembershipType", StringType(), True),
    StructField("JoinedDate", StringType(), True)

])

prod_schema = StructType([
    StructField("ProductID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("ProductCategory", StringType(), True),
    StructField("CostPrice", StringType(), True),
    StructField("RetailPrice", StringType(), True)
    
])

sales_schema = StructType([
    StructField("TransactionID", StringType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("QuantityPurchased", StringType(), True)
])

df_cust_bronze = spark.read.format("csv").option("header", "true").schema(cust_schema).load("Files/bronze/retail_customers.csv")
df_prod_bronze = spark.read.format("csv").option("header","true").schema(prod_schema).load("Files/bronze/retail_products.csv")
df_sales_bronze = spark.read.format("csv").option("hrader", "true").schema(sales_schema).load("Files/bronze/retail_sales.csv")

print("📥 bronze Layer Data Loaded Successfully into Spark Memory!")

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 7, Finished, Available, Finished, False)

📥 bronze Layer Data Loaded Successfully into Spark Memory!


In [5]:
df_prod_bronze.show(5)

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 8, Finished, Available, Finished, False)

+---------+-------------------+---------------+---------+-----------+
|ProductID|        ProductName|ProductCategory|CostPrice|RetailPrice|
+---------+-------------------+---------------+---------+-----------+
|   PRD901|        Smart Watch|    Electronics|     2500|       4999|
|   PRD902|Mechanical Keyboard|    Accessories|     1500| text_error|
|   PRD903|      Running Shoes|        Fashion|     1200|      -2000|
|   PRD904|       Coffee Maker|Home Appliances|     3000|       5500|
|   PRD905|  Bluetooth Speaker|    Electronics|     1800|       3499|
+---------+-------------------+---------------+---------+-----------+
only showing top 5 rows



In [6]:
display(df_cust_bronze.limit(5))

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ab0c097e-dc9a-4191-9779-70ec48b2b111)

In [7]:
from pyspark.sql.functions import trim, upper, to_date, row_number, when, monotonically_increasing_id, lower,   concat,lit
from pyspark.sql.window import Window

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 10, Finished, Available, Finished, False)

In [8]:
df_cust_clean =df_cust_bronze \
    .withColumn("CustomerName", trim(col("CustomerNAme"))) \
    .withColumn("City", trim(col("City"))) \
    .withColumn("MembershipType", upper(trim(col=("MembershipType")))) \
    .withColumn("JoinedDate", to_date(col("JoinedDate"), "yyyy-MM-dd")) \
    .fillna({"CustomerName": "Unknown", "City": "Unknown"})

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 11, Finished, Available, Finished, False)

In [9]:
display(df_cust_clean.limit(5))

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c43558c6-7fce-42fb-b49b-01733d5f338c)

In [10]:
window_spec_cust = Window.partitionBy("CustomerID").orderBy(col("JoinedDate").desc())

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 13, Finished, Available, Finished, False)

In [11]:

df_cust_dedup = df_cust_clean \
    .withColumn("row_num", row_number().over(window_spec_cust)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 14, Finished, Available, Finished, False)

In [12]:
df_cust_silver = df_cust_dedup \
    .withColumn("Customer_SK", monotonically_increasing_id()) \
    .withColumn("Email", lower(concat(col("CustomerID"), lit("@retail.com"))))

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 15, Finished, Available, Finished, False)

In [13]:
df_prod_cast = df_prod_bronze \
    .withColumn("CostPrice", col("CostPrice").cast("double")) \
    .withColumn("RetailPrice", col("RetailPrice").cast("double")) 


StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 16, Finished, Available, Finished, False)

In [14]:
valid_prod_condition = (col("CostPrice").isNotNull()) & \
                       (col("RetailPrice").isNotNull()) & \
                       (col("RetailPrice") > 0) & \
                       (col("CostPrice") > 0)

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 17, Finished, Available, Finished, False)

In [15]:
df_prod_silver = df_prod_cast.filter(valid_prod_condition).withColumn("product_SK", monotonically_increasing_id())
df_prod_rejected = df_prod_cast.filter(~valid_prod_condition).withColumn("RejectionReason", lit("Invalid Price or Data Mismatch"))

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 18, Finished, Available, Finished, False)

In [16]:
valid_sales_condition = (col("QuantityPurchased").isNotNull()) & (col("QuantityPurchased") > 0)


StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 19, Finished, Available, Finished, False)

In [17]:
df_sales_silver = df_sales_bronze.filter(valid_sales_condition).withColumn("InvalidDate", to_date(col("InvoiceDate"), "yyyy-MM-dd"))
df_sales_rejected = df_sales_bronze.filter(~valid_sales_condition).withColumn("RejectReason", lit("Invalid Quantity Purchased"))

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 20, Finished, Available, Finished, False)

In [18]:
print("=== SILVER LAYER PROCESSING COMPLETED ===")
print(f"Valid Customers (Silver) : {df_cust_silver.count()}")
print(f"Valid Products (Silver) : {df_prod_silver.count()} | Rejected Products: {df_prod_rejected.count()}")
print(f"Valid Sales (Silver)    : {df_sales_silver.count()} | Rejected Sales: {df_sales_rejected.count()}")

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 21, Finished, Available, Finished, False)

=== SILVER LAYER PROCESSING COMPLETED ===
Valid Customers (Silver) : 50
Valid Products (Silver) : 48 | Rejected Products: 2
Valid Sales (Silver)    : 50 | Rejected Sales: 0


In [20]:



df_cust_silver.write.format("delta").mode("overwrite").save("Files/silver/silver_customers")
df_prod_silver.write.format("delta").mode("overwrite").save("Files/silver/silver_products")
df_sales_silver.write.format("delta").mode("overwrite").save("Files/silver/silver_sales")


df_prod_rejected.write.format("delta").mode("overwrite").save("Files/rejected/rejected_products")
df_sales_rejected.write.format("delta").mode("overwrite").save("Files/rejected/rejected_sales")

print("💾 Cell 3 Success: Silver Data and Rejected Logs are safely written in DELTA format!")


StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 24, Finished, Available, Finished, False)

💾 Cell 3 Success: Silver Data and Rejected Logs are safely written in DELTA format!


In [24]:
from pyspark.sql.functions import sum, count, broadcast


silver_cust = spark.read.format("delta").load("Files/silver/silver_customers")
silver_prod = spark.read.format("delta").load("Files/silver/silver_products")
silver_sales = spark.read.format("delta").load("Files/silver/silver_sales")


df_enriched_gold = silver_sales \
    .join(broadcast(silver_cust), "CustomerID", "inner") \
    .join(broadcast(silver_prod), "ProductID", "inner")

df_enriched_gold.cache()

df_gold_metrics = df_enriched_gold \
    .withColumn("Total_Revenue", col("QuantityPurchased") * col("RetailPrice").cast("double")) \
    .withColumn("Total_Cost", col("QuantityPurchased") * col("CostPrice").cast("double")) \
    .withColumn("Net_Profit", col("Total_Revenue") - col("Total_Cost"))

final_gold_report = df_gold_metrics.groupBy(
    "City", "MembershipType", "ProductCategory", "ProductName", "InvoiceDate", "CustomerName"
).agg(
    count("TransactionID").alias("Total_Orders"),
    sum("QuantityPurchased").alias("Total_Items_Sold"),
    sum("Total_Revenue").alias("Total_Revenue"),
    sum("Net_Profit").alias("Total_Profit")
)

final_gold_report.write.format("delta").mode("overwrite").saveAsTable("gold_business_metrics")

df_enriched_gold.unpersist()

print("🏆 Gold Business Metrics Table Created and Saved in DELTA format!")
final_gold_report.show(5)


StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 36, Finished, Available, Finished, False)

🏆 Gold Business Metrics Table Created and Saved in DELTA format!
+---------+--------------+---------------+---------------+-----------+----------------+------------+----------------+-------------+------------+
|     City|MembershipType|ProductCategory|    ProductName|InvoiceDate|    CustomerName|Total_Orders|Total_Items_Sold|Total_Revenue|Total_Profit|
+---------+--------------+---------------+---------------+-----------+----------------+------------+----------------+-------------+------------+
|   London|      PLATINUM|    Electronics|Fitness Tracker| 2026-02-07|    David Miller|           1|             1.0|       4500.0|      2300.0|
|   Boston|          GOLD|        Fashion| Leather Wallet| 2026-02-20|Barbara Robinson|           1|             2.0|       3000.0|      1400.0|
|  Atlanta|        SILVER|    Electronics|     Power Bank| 2026-02-27|      Mark Allen|           1|             1.0|       1999.0|       999.0|
|  Detroit|      PLATINUM|    Accessories| Wireless Mouse| 2026-0

In [25]:
display(final_gold_report.limit(5))

StatementMeta(, dfec0ecf-8476-4eb1-b327-16cce4b9a2bc, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6befdda7-48c2-4346-80c2-a2537f1a814f)